In [1]:
# Lab type: prompt
# Course: AI401 — AI Applications with LLMs
# Lesson: Oversight Systems: Designing the Human-in-the-Loop Interface
# Task: Use an AI coding tool to implement a review queue routing function, then audit the result against production criteria

# Lab: Prompting for and Auditing Review Queue Routing Logic

## Scenario

Your team is building a medical triage classification system (Risk Tier 2). An LLM classifies incoming patient messages into one of five categories: `urgent`, `appointment`, `medication`, `results`, `admin`. The classifier passes all three validation layers most of the time — but Tier 2 requires routing uncertain or anomalous outputs to human review before the classification is acted on.

You need to implement `should_route_to_review()`. Your task has three parts:

1. **Write a prompt** for an AI coding assistant to implement the function
2. **Paste and run the AI-generated code** in this notebook
3. **Audit the result** against the checklist below

## Provided context — the data structures

In [2]:
from dataclasses import dataclass, field
from typing import Literal


MedicalCategory = Literal['urgent', 'appointment', 'medication', 'results', 'admin']


@dataclass
class ClassificationOutput:
    category: MedicalCategory
    confidence: float          # 0.0 – 1.0, higher is more confident
    output_tokens: int         # number of tokens in the LLM response


@dataclass
class ReviewQueueItem:
    item_id: str
    input_summary: str                      # human-readable patient message summary
    llm_output: ClassificationOutput
    validation_result: Literal['pass', 'fail']  # schema + semantic + behavioural
    anomaly_flags: list[str] = field(default_factory=list)
    # e.g. ['length', 'ood_input', 'boundary_date']
    input_similarity_to_golden_set: float = 1.0  # 0–1, lower = more OOD


# Example items
examples = [
    ReviewQueueItem(
        item_id='item_001',
        input_summary='chest pain, shortness of breath since this morning',
        llm_output=ClassificationOutput('urgent', confidence=0.95, output_tokens=12),
        validation_result='pass',
        anomaly_flags=[],
        input_similarity_to_golden_set=0.88,
    ),
    ReviewQueueItem(
        item_id='item_002',
        input_summary='need prescription refilled soon',
        llm_output=ClassificationOutput('medication', confidence=0.61, output_tokens=14),
        validation_result='pass',
        anomaly_flags=[],
        input_similarity_to_golden_set=0.72,
    ),
    ReviewQueueItem(
        item_id='item_003',
        input_summary='what are your office hours',
        llm_output=ClassificationOutput('admin', confidence=0.89, output_tokens=3),
        validation_result='pass',
        anomaly_flags=['length'],
        input_similarity_to_golden_set=0.91,
    ),
    ReviewQueueItem(
        item_id='item_004',
        input_summary='patient sent 4000-word message in a foreign language',
        llm_output=ClassificationOutput('admin', confidence=0.43, output_tokens=8),
        validation_result='fail',
        anomaly_flags=['ood_input', 'length'],
        input_similarity_to_golden_set=0.21,
    ),
]

print(f'Loaded {len(examples)} example ReviewQueueItems')

Loaded 4 example ReviewQueueItems


## Step 1: Write your prompt

Write a prompt in the cell below that you would give to an AI coding assistant to implement `should_route_to_review(item: ReviewQueueItem) -> tuple[bool, str]`.

The function should return `(True, reason)` when the item should go to human review, and `(False, 'auto_approved')` otherwise.

**Before writing:** consider what routing rules the lesson says are appropriate for a Tier 2 application. Your prompt should be specific enough that the AI can implement all required routing criteria without guessing.

In [3]:
# Write your prompt here as a Python string
my_prompt = """

"""

<details>
<summary>🔑 Model prompt — Task 1</summary>

**Example strong prompt:**

> Implement the Python function `should_route_to_review(item: ReviewQueueItem, confidence_threshold: float = 0.7) -> tuple[bool, str]` using the dataclasses defined above.
>
> Return `(True, reason)` when ANY of the following routing rules applies:
> 1. `item.validation_result == 'fail'` → reason: `'validation_failed'`
> 2. `item.llm_output.confidence < confidence_threshold` → reason: `'low_confidence'`
> 3. Any flag in `item.anomaly_flags` is one of `{'length', 'repetition', 'ood'}` → reason: `'anomaly:<flag>'` (use the first matching flag)
> 4. `item.input_similarity_to_golden_set < 0.5` → reason: `'ood_input'`
>
> Otherwise return `(False, 'auto_approved')`.
>
> Requirements:
> - `confidence_threshold` must be a function parameter with default `0.7` — do not hardcode it in the function body
> - Unknown anomaly flags must be handled gracefully without raising `KeyError` or crashing
> - Always return a `tuple[bool, str]` — never return a bare `bool`
> - No imports beyond what is already defined in this notebook

**Why it's strong:** It enumerates every routing rule explicitly so the AI has no room to guess, names the exact return type and reason strings, calls out the two most common AI failure modes (hardcoded threshold, crash on unknown flag) as explicit constraints, and scopes imports to prevent the AI from introducing unnecessary dependencies.

</details>

## Step 2: Paste and run the AI-generated implementation

Run your prompt through an AI coding assistant (Claude, Copilot, etc.), then paste the generated function below and run it.

In [4]:
# Paste the AI-generated implementation here

def should_route_to_review(item: ReviewQueueItem) -> tuple[bool, str]:
    """Paste AI-generated implementation here."""
    raise NotImplementedError('Replace this with the AI-generated function')

In [5]:
# Run the function on the example items
for item in examples:
    try:
        route, reason = should_route_to_review(item)
        print(f'{item.item_id}  route={route}  reason={reason!r}')
    except NotImplementedError:
        print('Paste the AI-generated implementation in step2_002 first')
        break

Paste the AI-generated implementation in step2_002 first


## Step 3: Audit the AI-generated implementation

Answer each checklist item below. For each one, write:
- `PASS` if the criterion is met
- `FAIL` with a brief explanation if it is not
- `PARTIAL` if it is partially met

Then fix any failures in the cell at the end.

In [6]:
# Audit criterion 1:
# Does the function route items where validation_result == 'fail'?
# Expected: item_004 (validation_result='fail') → route=True
for item in examples:
    if item.validation_result == 'fail':
        try:
            route, reason = should_route_to_review(item)
            print(f'{item.item_id}: route={route}, reason={reason!r}')
            print(f'Criterion 1: {"PASS" if route else "FAIL — validation failure not routed"}')
        except NotImplementedError:
            print('Implement the function first')

Implement the function first


In [7]:
# Audit criterion 2:
# Does the function route items with confidence < 0.7?
# Expected: item_002 (confidence=0.61) → route=True
for item in examples:
    if item.llm_output.confidence < 0.7 and item.validation_result == 'pass':
        try:
            route, reason = should_route_to_review(item)
            print(f'{item.item_id}: confidence={item.llm_output.confidence}, '
                  f'route={route}, reason={reason!r}')
            print(f'Criterion 2: {"PASS" if route else "FAIL — low confidence not routed"}')
        except NotImplementedError:
            print('Implement the function first')

Implement the function first


In [8]:
# Audit criterion 3:
# Does the function route items with 'length' in anomaly_flags?
# Expected: item_003 (anomaly_flags=['length']) → route=True
for item in examples:
    if 'length' in item.anomaly_flags and item.validation_result == 'pass':
        try:
            route, reason = should_route_to_review(item)
            print(f'{item.item_id}: anomaly_flags={item.anomaly_flags}, '
                  f'route={route}, reason={reason!r}')
            print(f'Criterion 3: {"PASS" if route else "FAIL — anomalous length not routed"}')
        except NotImplementedError:
            print('Implement the function first')

Implement the function first


In [9]:
# Audit criterion 4:
# Does the function handle unknown anomaly flags without crashing?
unknown_flag_item = ReviewQueueItem(
    item_id='item_005',
    input_summary='test with unknown anomaly flag',
    llm_output=ClassificationOutput('admin', confidence=0.85, output_tokens=10),
    validation_result='pass',
    anomaly_flags=['new_unknown_flag_type'],
    input_similarity_to_golden_set=0.90,
)
try:
    route, reason = should_route_to_review(unknown_flag_item)
    print(f'Handled unknown flag: route={route}, reason={reason!r}')
    print('Criterion 4: PASS (no exception raised)')
except NotImplementedError:
    print('Implement the function first')
except Exception as e:
    print(f'Criterion 4: FAIL — raised {type(e).__name__}: {e}')

Implement the function first


In [10]:
# Audit criterion 5:
# Is the confidence threshold parameterised (not hardcoded)?
# Inspect the function source to check
import inspect
try:
    src = inspect.getsource(should_route_to_review)
    has_default_param = 'confidence_threshold' in src or 'threshold' in src
    has_hardcoded = '0.7' in src or '0.65' in src or '0.75' in src
    print('Source snippet (routing logic):')
    for line in src.splitlines()[1:10]:
        print(' ', line)
    print()
    if has_default_param:
        print('Criterion 5: PASS — threshold is a parameter')
    elif has_hardcoded:
        print('Criterion 5: PARTIAL — threshold is hardcoded; '
              'consider making it a parameter for per-deployment tuning')
    else:
        print('Criterion 5: inconclusive — review the source above manually')
except (NotImplementedError, OSError):
    print('Implement the function first')

Source snippet (routing logic):
      """Paste AI-generated implementation here."""
      raise NotImplementedError('Replace this with the AI-generated function')

Criterion 5: inconclusive — review the source above manually


## Step 4: Fix any failures and reflect

Rewrite `should_route_to_review` below, addressing the criteria your audit found failing.

In [13]:
def should_route_to_review_v2(
    item: ReviewQueueItem,
    confidence_threshold: float = 0.7,
) -> tuple[bool, str]:
    """Revised implementation addressing audit findings."""
    # Placeholder implementation to prevent TypeError in step4_003
    # Replace with actual routing logic based on audit findings
    return False, 'auto_approved' # Example: always auto-approve for now

<details>
<summary>🔑 Expected AI failure modes — Task 4</summary>

These are the failure modes AI coding tools most commonly introduce in this task. Check your audit findings against this list.

**Error 1 — Hardcoded confidence threshold:** The threshold (`0.7` or similar) is embedded as a magic number in the function body rather than a parameter. This makes it impossible to adjust routing sensitivity without editing the source, and automated tests cannot vary the threshold.

**Error 2 — Unknown anomaly flags crash or silently pass:** Either a `KeyError` is raised for flags not in the expected set, or unknown flags are silently ignored and the item is auto-approved when it should be routed for human review. The safe approach is to route on any unknown flag (conservative) or log and skip (if only known flags are actionable).

**Error 3 — `input_similarity_to_golden_set` omitted:** The OOD-input signal is frequently left out of the generated implementation. Without it, items with inputs unlike anything in the training/golden set pass straight through to auto-approval — a significant gap for a Tier 2 medical application.

**Error 4 — Return type is bare `bool`:** The function returns `True`/`False` instead of `(bool, str)`. The `reason` string is the primary audit signal — without it, reviewers cannot see why an item was routed, making the queue unactionable.

**Error 5 — High-confidence validation failures auto-approved:** Items with `confidence=0.95` and `validation_result='fail'` are incorrectly auto-approved because the confidence check passes. Validation-layer failures must always route to human review, independent of confidence score.

</details>

In [14]:
# Run all examples through your revised function
print('Results from should_route_to_review_v2:')
for item in examples:
    try:
        route, reason = should_route_to_review_v2(item)
        print(f'  {item.item_id}  route={route}  reason={reason!r}')
    except NotImplementedError:
        print('  Implement should_route_to_review_v2 above')
        break

Results from should_route_to_review_v2:
  item_001  route=False  reason='auto_approved'
  item_002  route=False  reason='auto_approved'
  item_003  route=False  reason='auto_approved'
  item_004  route=False  reason='auto_approved'


In [15]:
# Instructor note — expected AI failure modes (hidden: True)
# 1. Confidence threshold hardcoded, not parameterised
# 2. Unknown anomaly flags may raise KeyError or be silently ignored
# 3. OOD input signal (input_similarity_to_golden_set) frequently omitted
# 4. Return type may be bool only, not (bool, reason) — reduces auditability
# 5. High-confidence incorrect outputs (high confidence + validation fail) may be auto-approved
pass